## Load Seaflow dataset
Version 1.6 ([Ribalet et al. 2024](https://zenodo.org/records/10896099))

In [3]:
# import libraries
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter('ignore')

new_sf=pd.read_parquet('../data/seaflow_1.6.parquet')
## cleaning
# set tz to UTC
new_sf['time']=pd.DatetimeIndex(new_sf['time']).tz_localize('UTC')
new_sf['day_year']=new_sf['time'].apply(lambda x: x.timetuple().tm_yday)
## define rows that are within Station ALOHA region
new_sf['ALOHA']=False
new_sf.loc[(new_sf['lat'] >= 22.25) &
               (new_sf['lat'] <= 23.25) & 
               (new_sf['lon'] <= -157.5) &
               (new_sf['lon'] >= -158.5), 'ALOHA']=True
# offset the longitude for international dateline rounding
new_sf['lon']=np.where(new_sf['lon']>0, new_sf['lon'], new_sf['lon']+360)

## melt to long
#double melt, first by abundance
temp_sf=new_sf.rename(columns={'abundance_prochloro':'prochloro',
                                  'abundance_synecho':'synecho',
                                 'abundance_picoeuk':'picoeuk'})
abund_sf=pd.melt(temp_sf, id_vars=['time','day_year','cruise','lat','lon', 'ALOHA'],
        var_name='pop',
        value_vars=['prochloro', 'synecho','picoeuk'],
        value_name='abundance')
# melt by Qc
temp_sf=new_sf.rename(columns={'Qc_prochloro':'prochloro',
                                  'Qc_synecho':'synecho',
                                 'Qc_picoeuk':'picoeuk'})
qc_sf=pd.melt(temp_sf, id_vars=['time','day_year','cruise','lat','lon', 'ALOHA'],
        var_name='pop',
        value_vars=['prochloro', 'synecho','picoeuk'],
        value_name='Qc_hour')
# by diam
temp_sf=new_sf.rename(columns={'diam_prochloro':'prochloro',
                                  'diam_synecho':'synecho',
                                 'diam_picoeuk':'picoeuk'})
diam_sf=pd.melt(temp_sf, id_vars=['time','day_year','cruise','lat','lon', 'ALOHA'],
        var_name='pop',
        value_vars=['prochloro', 'synecho','picoeuk'],
        value_name='diam_hour')

# merge Qc and abundance dfs
sf_merge1=abund_sf.merge(qc_sf)
# merge again with diam 
sf_merge2=sf_merge1.merge(diam_sf)
# remove abundance that's too low
sf_clean=sf_merge2[sf_merge2['abundance']> 0.02]

In [5]:
import os
from astral import Observer
import sys
sys.path.insert(0,'/Users/Kathy/Desktop/UW/seaflow/decomposition_project/scripts/')
from diel_tools_clean import sunrise_sunset, label_daytime

# check if file exists, create hourly_sf if not
filepath = '../data/hourly_sf_v1_all.pickle'

if os.path.exists(filepath):
    # open hourly_sf file
    hourly_sf = pd.read_pickle('../data/hourly_sf_v1_all.pickle')
else:
    print(f"'{filepath}' does not exist. Creating file...")
    # create observer col
    sf_clean['obs']=sf_clean.apply(lambda x: Observer(x.lat, x.lon, 0), axis=1)

    # create hourly rounded sunrise/sunset cols for sf_clean 
    res=sf_clean.apply(lambda x: sunrise_sunset(x.time, x.obs), axis=1)
    sf_clean[['sunrise', 'sunset']]=pd.DataFrame(res.tolist(), index=sf_clean.index)
    # aggregate and choose one sunrise/sunset value (max value of time)
    hourly_sf=sf_clean.groupby([pd.Grouper(key='time',freq='1H'),
                    'pop','cruise']).agg({
        'lat':'mean',
        'lon':'mean',
        'ALOHA':'mean',
        'abundance':'mean',
        'Qc_hour':'mean',
        'sunrise':'max',
        'sunset':'max'
    }).reset_index()

    # apply to dataframe
    # annette has 8817 after filtering (8889 if no filtering) and excluding croco (i have 8874)
    hourly_sf['night']=hourly_sf.apply(lambda x: label_daytime(x.time, x.sunrise, x.sunset), axis=1)
    hourly_sf['par']=0
    # save data 
    hourly_sf.to_pickle('../data/hourly_sf_v1_all.pickle')

## Run STL and 3-Day Rolling models
- Run on prochloro, synecho, picoeuk

In [6]:
from tsd_functions_clean import run_full_model

# load files if they exist
# LOAD data from saved files
tsd_file = '../data/all_seaflow_tsd_mixed_v1_all.pickle'
rates_file = '../data/all_rates_mixed_v1_all.pickle'
fails_file = '../data/failed_days_mixed_v1_all.pickle'
final_file = '../data/final_rates_v1_all.pickle'

## all files must exist or run model
if (os.path.exists(tsd_file)) & (os.path.exists(rates_file)) & (os.path.exists(fails_file)) & (os.path.exists(final_file)):
    # open files
    all_tsd = pd.read_pickle(tsd_file)
    daily_rates = pd.read_pickle(rates_file)
    # get daily data
    mean_tsd=all_tsd.groupby(['cruise','cruise_day','pop'])[['lat','lon','pop']].mean().reset_index()
    # merge
    merge_rates=daily_rates.merge(mean_tsd)
    failed_days_df = pd.read_pickle(fails_file)
    # remove days that have failed to run from tsd df
    merge_tsd=all_tsd.merge(failed_days_df[['cruise','cruise_day','pop','model']], 
                        how='outer', indicator=True)
    good_tsd=merge_tsd[merge_tsd['_merge']=='left_only']
    final_rates=pd.read_pickle(final_file)
# run model for each cruise and population
else:
    pops=['prochloro','synecho','picoeuk']
    # save dataframes
    growth_rates=[]
    tsd_results=[]
    impute_dfs = []
    all_names=pd.unique(hourly_sf['cruise'])
    # cruises that fail to run due to being too short
    failed_cruises=[]
    # days that fail to run due to not having enough data points
    failed_days=[]
    for cruise_name in all_names:
        # subset by cruise
        print(cruise_name)
        cruise_df=hourly_sf.loc[hourly_sf['cruise']==cruise_name]
        for pop in pops:
            print(pop)
            # subset df by dataframe
            pop_df=cruise_df.loc[cruise_df['pop']==pop]
            # add column for missing data to be filled
            pop_df['data_with_missing']=pop_df['Qc_hour']
            # run full model (V1)
            impute_df, tsd_df, growth, skip_df=run_full_model(df=pop_df.drop(columns=['sunrise','sunset']), 
                                            col='Qc_hour', missing_col='data_with_missing',pop=pop)
            # keep going if failed
            if tsd_df is None:
                # save pop and cruise
                failed_cruises.append({cruise_name:pop})
                continue
            # save results
            impute_dfs.append(impute_df)
            growth_rates.append(growth)
            tsd_results.append(tsd_df)
            failed_days.append(skip_df)

    # save data
    impute_results = pd.concat(impute_dfs)
    impute_results.to_pickle('../data/imputed_results.pickle')

    all_tsd=pd.concat(tsd_results)
    # # get daily data
    mean_tsd=all_tsd.groupby(['cruise','cruise_day','pop'])[['lat','lon','pop', 'biomass']].mean().reset_index()
    daily_rates=pd.concat(growth_rates)
    # merge
    merge_rates=daily_rates.merge(mean_tsd)
    failed_days_df=pd.concat(failed_days)
    merge_tsd=all_tsd.merge(failed_days_df[['cruise','cruise_day','pop','model']], 
                        how='outer', indicator=True)
    good_tsd=merge_tsd[merge_tsd['_merge']=='left_only']

## Quality Control
Don't need to run this step if you already have the final rates file loaded

1) Check for autocorrelation
2) Daily growth has a significant (p < 0.01) slope

1) Remove autocorrelated residuals

In [7]:
from model_qc import check_resids

# store autocorrelated residuals
auto_resids=[]
# loop through each cruise day and model for each cruise
for cruise in pd.unique(good_tsd['cruise']):
    cruise_df=good_tsd.loc[good_tsd['cruise']==cruise]
    # loop through each pop
    for pop in pd.unique(cruise_df['pop']):
        pop_df=cruise_df.loc[cruise_df['pop']==pop]
        # loop thru each model
        for model in pd.unique(pop_df['model']):
            model_df=pop_df.loc[pop_df['model']==model]
            # significant residuasl
            bad_residuals=check_resids(model_df)
            # save days with bad residuals as df
            auto_resids.append(model_df.loc[model_df['cruise_day'].isin(bad_residuals)])

# save bad residuals
bad_resids_df=pd.concat(auto_resids)
# let's remove the bad residuals from the growth rate df
sum_bad_resids=bad_resids_df[['cruise','cruise_day','pop','model','lat','lon']].groupby(
    ['cruise','cruise_day','pop', 'model']).mean().reset_index()        

# merge back with rates
good_rates=merge_rates.merge(sum_bad_resids, on=['cruise','cruise_day','pop','model', 'lat','lon'],
                             how='outer', indicator=True)
good_rates=good_rates[good_rates['_merge']=='left_only']

2. Check for significance in daily growth

In [8]:
sig_rates=good_rates.loc[(good_rates['pval']<0.01)&(good_rates['daily_growth']>0)]
bad_rates=good_rates.loc[(good_rates['pval']>0.01)|(good_rates['daily_growth']<0)]

3. Model selection based on RMSE*

*This is also the same as taking the square root of the residual for each model

In [ ]:
from model_qc import r2_rmse
# grab good days from tsd df
sum_sig_rates=sig_rates[['cruise','cruise_day','pop','model']]
sig_tsd=sum_sig_rates.merge(good_tsd)
# calculate the RMSE for each day and model
group_rmse=sig_tsd.groupby(['cruise','cruise_day','pop','model']).apply(r2_rmse).reset_index()
# get minimum rmse per model on each day
good_models=group_rmse.groupby(['cruise','cruise_day','pop']).agg({
    'rmse':'min'
}).reset_index()
# retrieve model infromation from first grouped df
good_models=good_models.merge(group_rmse)
# filter good rates from good models
final_rates=sig_rates.merge(good_models)
final_rates=final_rates.loc[final_rates['productivity']<20]
# save final rates
final_rates.to_pickle('../data/final_rates_v1_all.pickle')